# 06 · Data mixtures, sampling temperature and epochs

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nilesh-salpe/visual-ai/blob/main/build-it/pretraining-data/06_data_mixture.ipynb) &nbsp; 📖 Concept deck: [Data mixtures and repetition](https://nilesh-salpe.github.io/visual-ai/pretraining-data/09-mixtures-and-repetition/) &nbsp; ⏱️ Suggested time: **25 min**

**You will build:** turning domain weights into a token budget, computing how many epochs each domain is repeated, temperature sampling to up-weight small domains (as in multilingual training), and capping repetition.

**How to use this notebook**
1. Run the **Setup** cell.
2. For each task, fill in the `TODO` in the code cell, then run the **Tests** cell under it. You'll see ⏳ (not done yet), ✅ (passed) or ❌ (failed, with the reason).
3. Stuck? The **Reference solutions** at the bottom are collapsed. Try first.

Everything runs on CPU with NumPy, so no GPU is needed.

In [ ]:
# @title Setup: run this first
import numpy as np, math, time, itertools, collections
np.set_printoptions(precision=4, suppress=True)

def check(name, test, impl):
    """Run `test(impl)` and report: ⏳ not implemented, ✅ passed, ❌ failed with the reason."""
    try:
        test(impl)
        print(f"✅ {name}: all tests passed")
    except NotImplementedError:
        print(f"⏳ {name}: not implemented yet. Fill in the TODO in the cell above, then re-run this cell.")
    except AssertionError as e:
        print(f"❌ {name}: {e}")


DOMAINS = {"web": 12_000e9, "code": 800e9, "math": 150e9, "books": 60e9, "wikipedia": 20e9, "low-resource languages": 5e9}

## Task 1: Epochs per domain

Given unique tokens per domain, sampling weights (summing to 1) and a total training budget, return a dict of **epochs** per domain: weight × budget ÷ unique tokens.

In [ ]:
def epochs(domains, weights, budget):
    # TODO
    raise NotImplementedError

In [ ]:
# @title Tests for Task 1: run after implementing
def _test_epochs(f):
    e = f({"a": 100, "b": 10}, {"a": 0.5, "b": 0.5}, 100)
    assert e == {"a": 0.5, "b": 5.0}

check("epochs", _test_epochs, epochs)

## Task 2: Temperature sampling

Natural sampling uses weights $p_i = n_i / \sum n$. Temperature sampling uses $q_i \propto p_i^{1/T}$: T = 1 is natural, larger T flattens toward uniform (more weight to small domains). Return the normalised weights.

In [ ]:
def temperature_weights(domains, T):
    # TODO
    raise NotImplementedError

In [ ]:
# @title Tests for Task 2: run after implementing
def _test_temperature_weights(f):
    w1 = f(DOMAINS, 1.0); assert abs(sum(w1.values()) - 1) < 1e-12
    assert abs(w1["web"] - 12000 / sum(v / 1e9 for v in DOMAINS.values())) < 1e-9
    w5 = f(DOMAINS, 5.0)
    assert w5["low-resource languages"] > 10 * w1["low-resource languages"], "higher T boosts small domains"

check("temperature_weights", _test_temperature_weights, temperature_weights)

## Task 3: Cap repetition

Muennighoff et al. (2023) found that up to about **4 epochs** of repeated data is nearly as good as fresh data, with rapidly diminishing returns after. Given desired weights, cap each domain at `max_epochs` and give the freed budget to uncapped domains in proportion to their weights; repeat until nothing exceeds the cap. Return the new weights.

In [ ]:
def cap_epochs(domains, weights, budget, max_epochs=4.0):
    # TODO: iterative water-filling
    raise NotImplementedError

In [ ]:
# @title Tests for Task 3: run after implementing
def _test_cap_epochs(f):
    tot = sum(DOMAINS.values()); q = {k: (v / tot) ** 0.2 for k, v in DOMAINS.items()}; w = {k: v / sum(q.values()) for k, v in q.items()}
    budget = 15_000e9
    c = f(DOMAINS, w, budget)
    e = {k: c[k] * budget / DOMAINS[k] for k in DOMAINS}
    assert abs(sum(c.values()) - 1) < 1e-9
    assert max(e.values()) <= 4.0 + 1e-6, f"no domain above 4 epochs, got {max(e.values()):.2f}"
    assert c["web"] > w["web"], "freed budget flows to the big domain"

check("cap_epochs", _test_cap_epochs, cap_epochs)

---
## Reference solutions
Try the tasks first. Each cell below is collapsed in Colab: double-click it (or click "Show code") to reveal the solution. Running it also checks the reference against the same tests.

In [ ]:
# @title Solution for Task 1: Epochs per domain
def epochs_solution(domains, weights, budget):
    return {k: weights[k] * budget / domains[k] for k in domains}

check("epochs (reference)", _test_epochs, epochs_solution)

In [ ]:
# @title Solution for Task 2: Temperature sampling
def temperature_weights_solution(domains, T):
    tot = sum(domains.values()); q = {k: (v / tot) ** (1 / T) for k, v in domains.items()}; z = sum(q.values())
    return {k: v / z for k, v in q.items()}

check("temperature_weights (reference)", _test_temperature_weights, temperature_weights_solution)

In [ ]:
# @title Solution for Task 3: Cap repetition
def cap_epochs_solution(domains, weights, budget, max_epochs=4.0):
    w = dict(weights); capped = set()
    while True:
        over = {k for k in w if k not in capped and w[k] * budget / domains[k] > max_epochs + 1e-12}
        if not over: return w
        for k in over:
            w[k] = max_epochs * domains[k] / budget; capped.add(k)
        free = 1 - sum(w[k] for k in capped); rest = {k: weights[k] for k in w if k not in capped}; z = sum(rest.values())
        for k in rest: w[k] = free * rest[k] / z

check("cap_epochs (reference)", _test_cap_epochs, cap_epochs_solution)

---
## Experiments
**A 15T-token run.** Epochs per domain under natural sampling, temperature 3, and temperature 3 with a 4-epoch cap.

In [ ]:
budget = 15_000e9
for name, w in [("natural (T=1)", temperature_weights_solution(DOMAINS, 1)), ("T=3", temperature_weights_solution(DOMAINS, 3)),
                ("T=3, capped at 4 epochs", cap_epochs_solution(DOMAINS, temperature_weights_solution(DOMAINS, 3), budget))]:
    e = epochs_solution(DOMAINS, w, budget)
    print(name + ": " + ", ".join(f"{k} {w[k]:.1%} ({e[k]:.1f} ep)" for k in DOMAINS))

---
## Takeaways
* A data mix is a set of sampling weights; with a fixed budget, small domains get repeated many times.
* Temperature sampling up-weights small domains (common for languages); capping epochs (~4) avoids wasted repetition.
* Llama 3's final mix was roughly 50% general knowledge, 25% math and reasoning, 17% code and 8% multilingual tokens.